In [ ]:
import os, random
import cv2 as cv
from pathlib import Path

In [ ]:
### dict with class labels
class_lables = {
    "Brown bear": 0,
    "Canary": 1,
    "Cheetah": 2,
    "Crocodile": 3,
    "Elephant": 4,
    "Fox": 5,
    "Goat": 6,
    "Goldfish": 7,
    "Kangaroo": 8,
    "Leopard": 9,
    "Mouse": 10,
    "Ostrich": 11,
    "Panda": 12,
    "Pig": 13,
    "Rabbit": 14,
    "Raccoon": 15,
    "Rhinoceros": 16,
    "Sheep": 17,
    "Woodpecker": 18,
    "Zebra": 19,
}

### list with output folders 
output_folders = ['/images', '/labels', '/images/test', '/images/val', '/images/train', '/labels/test', '/labels/val', '/labels/train']

In [ ]:
#### Helper function for reading and transforming label files to correct yolo format and writing to output path
def transform_label_file(input_path, output_path, width, height):
    ### creating output variabel
    output = []
    ### reading input file
    with open(input_path, 'r') as f:
        for line in f:
            content = line.strip().split()
            
            ### check and adjust for extra whitespace in the class label for animals with space in className in the label file.
            i = 0
            label = content[0]
            if len(content) == 6:
                i = 1
                label = label + " " + content[1]

            ### transforming the values in the label file to yolo format
            x_center = round(((float(content[1+i]) + float(content[3+i])) / 2 ) / width, 4)
            y_center = round(((float(content[2+i]) + float(content[4+i])) / 2 ) / height, 4)
            image_width = round((float(content[3+i]) - float(content[1+i])) / width, 4)
            image_height = round((float(content[4+i]) - float(content[2+i])) / height, 4)
            ### creating the output for the new transformed label file
            output.append(str(class_lables.get(label)) + " " + str(x_center) + " " + str(y_center) + " " + str(image_width) + " " + str(image_height))
    
    ### writing the output to the new label file
    with open(output_path, "w") as f:
        for line in output:
            f.write(f"{line}\n")
            
### function to preprocess image dataset with random seed for different spilts.
def pre_process_images(input_path, output_path, random_seed):
    ### looping through all folders and files in the input path.       
    for root, dirs, file in os.walk(input_path):
        for d in dirs:
            ### skiping Label folders
            if d != "Label":
                ### reading all the images paths in a folder and adding them to a list for processing
                image_paths = []
                d_path = os.path.join(root, d)
                for file in os.listdir(d_path):
                    if file.endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        image_paths.append(file)

                ### using random shuffel with random seed to create different selections of the images for the datasets.
                random.seed(random_seed)
                random.shuffle(image_paths)                
                
                ### creating splits for partition of dataset
                split_test = int(len(image_paths)*0.1) + 1
                split_val = int(len(image_paths)*0.25) + 1
                split_traing = int(len(image_paths)*0.80) + 1
                
                ### checking if output folders exists, if not then create folders.
                if not(os.path.exists(output_path)): 
                    os.mkdir(output_path)
                for p in output_folders:
                    path = output_path + p
                    if not(os.path.exists(path)):
                        os.mkdir(path)            
                    
                ### creating count variabel for spliting of the datasets
                count = 0
                ### looping through all images paths
                for file in image_paths:
                    path = os.path.join(d_path, file)
                    img = cv.imread(path)

                    ### creating the paths for the prosseceing of the images
                    label_input_path = d_path + "/Label/" + Path(file).stem + ".txt"                 
                    file_path = d + str(count) + Path(path).suffix
                    label_path = d + str(count) + ".txt"

                    ### creating dataset splits by using count and partition splits, creating splits with about, testing 10%, validation 15%, training 55% per animal class.
                    if count < split_test:
                        ### writing transformed label and image file to output path
                        transform_label_file(label_input_path, os.path.join(output_path, "labels/test/" + label_path), img.shape[1], img.shape[0])
                        cv.imwrite(os.path.join(output_path, "images/test/" + file_path), img)
                    elif count >= split_test and count < split_val:
                        ### writing transformed label and image file to output path
                        transform_label_file(label_input_path, os.path.join(output_path, "labels/val/" + label_path), img.shape[1], img.shape[0])
                        cv.imwrite(os.path.join(output_path, "images/val/" + file_path), img)                    
                    elif count >= split_val and count < split_traing:
                        ### writing transformed label and image file to output path
                        transform_label_file(label_input_path, os.path.join(output_path, "labels/train/" + label_path), img.shape[1], img.shape[0])
                        cv.imwrite(os.path.join(output_path, "images/train/" + file_path), img)
                    elif count >= split_traing:
                        break
                    count += 1


In [ ]:
### variabels for input and output paths for images and labels
input_path = './data/pre/'
output_path_1 = './data/post/1'
output_path_2 = './data/post/2'
output_path_3 = './data/post/3'

### runing the preprocessing of the images and labels
pre_process_images(input_path, output_path_1, 10)
pre_process_images(input_path, output_path_2, 20)
pre_process_images(input_path, output_path_3, 30)